## Flight price prediction using complex models

## Objective

Continuing on my [previous notebook's](https://www.kaggle.com/code/ayushsharma0812/linear-reg-residual-analysis-96-2-r-2-on-test) work, the objective of the study is to train more complex models on top of the dataset, taking into account the EDA and feature engineering already done in the previous notebook.

## Summary of what you will find

1. Pipleines that transform raw data into processed data that is ready for predictive modelling. This will help in automating deployment of the predictive model.
2. Custom functions for enhancing reusability. Custom classes for column transformations, which help in making compact pipelines adressing all our needs.
3. Cyclical/periodical encoding for columns that show cyclical sorting. We will also take a look at how using periodical encoding helps in improving performance compared to using ordinal encoding.
4. We will try tree based predictive algorithms, look at feature importances, train voting regressors and deep learning models as well. Use random search and cross-validation to do hyper parameter tuning and assess generalizaition performance of the models.
5. After training and comparing 6 different models, we are able to achieve **98.3% R^2 on the test dataset**.

## Aim behind this project

Implement pipleline based transformations for transforming raw data to fully processed data ready for predictive modelling. Practice and experiment with more complex models.

**<u>PS:</u>** 
- Feel free to contact me if you have any doubts or feedback through comment section or my socials.
- Please upvote the notebook if you like it, as it would motivate me to write more projects like these.

## My Socials

Follow me on these platforms for more such content:  


LinkedIn: https://www.linkedin.com/in/ayush-sharma-660831125/  
X(Twitter): https://x.com/_ayush_sh

## Importing the required libraries

In [ ]:
# basic libraries
import pandas as pd
import numpy as np 

#visualization libraries
import matplotlib.pyplot as plt 
import seaborn as sns 

#model building libraries
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn import tree
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, VotingRegressor
from xgboost import XGBRegressor
from tensorflow import keras
import tensorflow as tf

#decision tree visualization
from graphviz import Source
from IPython.display import Image, display
import pydotplus

#extra libraries
from pathlib import Path
from time import strftime
import warnings
warnings.filterwarnings('ignore')

## Dataset pre-processing

### Loading the dataset

In [ ]:
df = pd.read_csv("Clean_Dataset.csv")

### Understanding the data

In [ ]:
df.shape 

There are total 3,00,153 records and 12 columns.

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

The columns can be described as followed:  
1. **Unnamed: 0**: A row count which uniquely identifies each flight plan.  
2. **airline**: Name of the airline company.    
3. **flight**: Plane's flight code for given journey.  
4. **source_city**: City from which the flight takes off.  
5. **departure_time**: Time of Departure.
6. **stops**: Number of stops between the source and destination city.
7. **arrival_time**: Time of Arrival.
8. **destination_city**: City where the flight will land.
9. **class**: Contains information on seat class.
10. **duration**: Total expected amount of the journey in hours.
11. **days_left**: Difference in days between the trip date and the date on which the data is extracted.
12. **price**: Ticket price, our target variable. 

The given columns can be divided into different categories based on the type of data that they hold, identifying this will guide the exploration that can be done while doing EDA and further processsing needed, if any, for predictive modelling:  
<u>Nomial categorical data</u>: airline, flight, source_city, departure_time, arrival_time, destination_city and seat class  
<u>Ordinal categorical data</u>: stops  
<u>Numerical data type</u>: Key(int), duration(float), days_left(int) and price(int) 

## Custom functions and classes

In [ ]:
class PowerTransformer(BaseEstimator, TransformerMixin): 
    '''To transform any given dataframe by applying a given power as an exponent.'''
    def __init__(self, power):
        self.power = power

    def fit(self, X, y=None):
        self.columns = X.columns
        return self

    def transform(self, X, y=None):
        return np.power(X, self.power)

class PeriodicTransformer(BaseEstimator, TransformerMixin):
    '''To transform ordinally encoded columns of a numpy array into periodically encoded features.'''
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        output = np.zeros((X.shape[0], 2 * X.shape[1]))
        for i in range(X.shape[1]):
            for j in range(2):
                if j == 0:
                    output[:, i*2 +j] = np.sin(2 * np.pi * X[:, i]/np.max(X[:, i]))
                else:
                    output[:, i*2 +j] = np.cos(2 * np.pi * X[:, i]/np.max(X[:, i]))
        return output

class PrintValTrainDiffCallback(tf.keras.callbacks.Callback):
    '''Defining new callback which will show the difference in train and validation r2 score at end of each epoch.'''
    def on_epoch_end(self, epoch, logs):
        diff = (logs["r2_score"] - logs["val_r2_score"])*100
        print(f" Diff. in r2 score(in %) = {diff:.2f}%")

def plot_decision_tree(model, feature_names, file_name):
    '''Plot decision tree graph from model'''
    tree_dot = tree.export_graphviz(model, out_file = None, rounded = True, filled = True, feature_names = feature_names)
    pydotplus.graph_from_dot_data(tree_dot).write_pdf(f'{file_name}.pdf')
    display(Image(pydotplus.graph_from_dot_data(tree_dot).create_png()))

def get_run_logdir(neurons, layers, lr, batch_size, root_logdir = 'my_logs'):
    '''Return path for log directory. This will help in generating new path for each model run.'''
    return f'{Path(root_logdir)}\\Run_{neurons}_{layers}_{lr}_{batch_size}'

def update_results_df(df, results, name):
    '''Update model-results dataframe with results from cross-validation.'''
    return df._append({'Model': name, 'Mean train score': np.mean(results['train_score']), 'Mean test score':np.mean(results['test_score'])}, ignore_index=True)

def plot_dl_model_history(history_obj):
    '''Plot history of the deep learning model on a suitable graph.'''
    pd.DataFrame(history_obj.history).plot(figsize = (8, 5), ylim = [0, 1], grid = True, xlabel = 'Epoch', style = ["r--", "r--.", "b-", "b-*"])
    plt.show();

## Train-Test split

I will follow a similar path to what I followed in the previous notebook, with minor tweaks(like cyclical encoding for arrival time and departure time) with a hope of improvement in performance.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(df.iloc[:, :-1], df.iloc[:, -1], test_size = 0.1, random_state = 123)

In [ ]:
df_train = pd.concat([X_train, Y_train], axis = 1)
df_test = pd.concat([X_test, Y_test], axis = 1)

In [ ]:
#freeing up memory
del X_train, X_test, Y_train, Y_test

## Train-Test split (based on seat classes)

In [ ]:
#segregating data based on seat class for train set generated earlier
df_train_eco = df_train[df_train['class'] == 'Economy'].drop(columns = ['class'])
df_train_bus = df_train[df_train['class'] == 'Business'].drop(columns = ['class'])

#generating copies of data to do manipulation on
df_train_eco_reg = df_train_eco.copy()
df_train_bus_reg = df_train_bus.copy()

## Common Pre-processing

### For economy class

**Few notes from previous notebook:**  
- The target variable was inverse of log10 of price.
- Duration feature was transformed by taking a cube root to make its distribution closer to a gaussian distribution, so that influence of data points with high predictor values in case of a linear regression model reduces.
- Arrival time and departure time have a periodic sorting within its categories and we would take that into account here.

Even though these complex models don't make any assumptions about the distribution of the inputs. We will continue with above transformations for an ample to ample comparison and because having a gaussian distrbution would help ensure that the algorithm don't focus on skewed points in any way. 

In [ ]:
#extracting input and target
X_train_eco_reg = df_train_eco_reg.iloc[:, :-1]
Y_train_eco_reg = np.power(np.log10(df_train_eco_reg.iloc[:, -1]), -1)

In [ ]:
#segregating columns based on pre-processing
columns = X_train_eco_reg.columns

#columns to be dropped
drop_cols = ['Unnamed: 0', 'flight']

#for numerical columns
cube_root_cols = ['duration']
num_cols = X_train_eco_reg.select_dtypes(include = ['float64', 'int64']).columns
passthrough_num_cols = num_cols.difference(cube_root_cols).difference(drop_cols)

#for categorical columns
ordinal_cols = ['stops']
oridnal_cols_cat_in_order = [['zero', 'one', 'two_or_more']]
periodic_cols = ['arrival_time', 'departure_time']
periodic_cols_cat_in_order = [['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night'], 
                          ['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night']]
cat_cols = X_train_eco_reg.select_dtypes(include = ['object']).columns
one_hot_encoding_cols = cat_cols.difference(ordinal_cols).difference(periodic_cols).difference(drop_cols)

In [ ]:
# For ordinal encoding of periodic columns first
eco_periodic_cols_ct = ColumnTransformer([("Ordinal_encoding", OrdinalEncoder(categories = periodic_cols_cat_in_order, dtype = int), periodic_cols)]) 

# pipeline for conversion of periodic columns first to ordinal encoding and then to cyclical encoding or periodic encoding
eco_periodic_pipeline = Pipeline([('ordinal_encoding', eco_periodic_cols_ct),
                              ('periodic_encoding', PeriodicTransformer())])

# column transformer for the rest of the data
eco_non_periodic_cols_ct = ColumnTransformer([('Cube_root', PowerTransformer(1/3), cube_root_cols),
                                          ('Passthrough_num_cols', 'passthrough', passthrough_num_cols),
                                          ('Ordinal_encoding', OrdinalEncoder(categories = oridnal_cols_cat_in_order, dtype = int), ordinal_cols),
                                          ('One_hot_encoding', OneHotEncoder(drop = 'first', sparse_output = False, dtype = int), one_hot_encoding_cols)])

#concatentation of data
eco_pipeline = FeatureUnion([("non_periodic_cols", eco_non_periodic_cols_ct), ("periodic_cols", eco_periodic_pipeline)])

In [ ]:
X_train_eco_reg_transformed = eco_pipeline.fit_transform(X_train_eco_reg)

The names for the columns get lost here in the above pipeline. Lets generate the column names based on the order of columns so that we can use that to better vizualise decision trees.

In [ ]:
eco_column_names = np.concatenate([[f'cube_root_{col}' for col in cube_root_cols],
                                   passthrough_num_cols,
                                   ordinal_cols,
                                   eco_non_periodic_cols_ct.named_transformers_['One_hot_encoding'].get_feature_names_out(), 
                                   [f'{col}_{trig}' for col in periodic_cols for trig in ['sin', 'cos']]], axis = 0)

In [ ]:
X_train_eco_reg_transformed = pd.DataFrame(X_train_eco_reg_transformed, columns = eco_column_names)

### For business class

**Few notes from previous notebook:**  
- The target variable was trasnformed by taking an inverse transformation with power of 0.3 of square root of price.
- Duration feature was transformed by taking a cube root to make its distribution closer to a gaussian distribution, so that influence of data points with high predictor values in case of a linear regression model reduces.  
- Arrival time and departure time have a periodic sorting within its categories and we would take that into account here.

In [ ]:
#extracting input and target
X_train_bus_reg = df_train_bus_reg.iloc[:, :-1]
Y_train_bus_reg = np.power(np.sqrt(df_train_bus_reg.iloc[:, -1]), -0.3)

In [ ]:
# For ordinal encoding of periodic columns first
bus_periodic_cols_ct = ColumnTransformer([("Ordinal_encoding", OrdinalEncoder(categories = periodic_cols_cat_in_order, dtype = int), periodic_cols)])

# pipeline for conversion of periodic columns first to ordinal encoding and then to cyclical encoding or periodic encoding
bus_periodic_pipeline = Pipeline([('ordinal_encoding', bus_periodic_cols_ct),
                              ('periodic_encoding', PeriodicTransformer())])

# column transformer for the rest of the data
bus_non_periodic_cols_ct = ColumnTransformer([('Cube_root', PowerTransformer(1/3), cube_root_cols),
                                          ('Passthrough_num_cols', 'passthrough', passthrough_num_cols),
                                          ('Ordinal_encoding', OrdinalEncoder(categories = oridnal_cols_cat_in_order, dtype = int), ordinal_cols),
                                          ('One_hot_encoding', OneHotEncoder(drop = 'first', sparse_output = False, dtype = int), one_hot_encoding_cols)])

#concatentation of data
bus_pipeline = FeatureUnion([("non_periodic_cols", bus_non_periodic_cols_ct), ("periodic_cols", bus_periodic_pipeline)])

In [ ]:
X_train_bus_reg_transformed = bus_pipeline.fit_transform(X_train_bus_reg)

In [ ]:
bus_column_names = np.concatenate([[f'cube_root_{col}' for col in cube_root_cols],
                                   passthrough_num_cols,
                                   ordinal_cols,
                                   bus_non_periodic_cols_ct.named_transformers_['One_hot_encoding'].get_feature_names_out(), 
                                   [f'{col}_{trig}' for col in periodic_cols for trig in ['sin', 'cos']]], axis = 0)

In [ ]:
X_train_bus_reg_transformed = pd.DataFrame(X_train_bus_reg_transformed, columns = bus_column_names)

## Decision tree model

### For economy class

To avoid too much computational complexity, I will only perform random search cv with 25 iterations and also, the splitter is sent to random.

#### Perform random search

In [ ]:
model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123)

param_grid = {'max_features': [None, 'sqrt', 'log2'],
              'min_samples_leaf': list(range(10, 101,10)),
              'max_depth' : list(range(1, 21, 2)),
              'criterion' :["squared_error", "friedman_mse", "absolute_error", "poisson"]}

random_cv = RandomizedSearchCV(model, param_distributions = param_grid, n_iter = 25, cv = 3, verbose = True, n_jobs = -1, scoring = 'r2' , return_train_score = False, random_state = 123)

best_random_cv = random_cv.fit(X_train_eco_reg_transformed, Y_train_eco_reg)

In [ ]:
best_random_cv.best_params_

#### Check overfitting

In [ ]:
model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123)
model.set_params(**best_random_cv.best_params_)

model_cv_result = cross_validate(model, X_train_eco_reg_transformed, Y_train_eco_reg, scoring='r2', cv=3, n_jobs=-1, verbose=0, return_train_score=True, return_estimator = False)

In [ ]:
eco_model_results_df = pd.DataFrame({'Model':['Decision Tree'], 'Mean train score': [np.mean(model_cv_result['train_score'])], 'Mean test score':[np.mean(model_cv_result['test_score'])]})

In [ ]:
eco_model_results_df

Little bit of overfitting. This can be made better by doing grid search around the obtained parameters.

#### Train model on full dataset

In [ ]:
eco_decision_tree_model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123).set_params(**best_random_cv.best_params_)
eco_decision_tree_model.fit(X_train_eco_reg_transformed, Y_train_eco_reg)

#### Decision tree visualization

In [ ]:
plot_decision_tree(eco_decision_tree_model, X_train_eco_reg_transformed.columns, 'Eco_decision_tree')

**Observations:**  
- Looking at the decision tree and observing the numerical columns, we can see multiple splits on these columns in sub branches of the tree. This means the model is trying to capture non-linearity. This we have already observed with linear regression as we needed multiple higher order polynomial terms in our model to improve performance.
- The performance is better than the linear regression model. This means there was more complexity within data compared to what we observed from EDA. This can be understood like this: what we modelled with polynomial and interatction terms in case of linear regression was limited to a bivaraite EDA(because of inability to visualize higher dimensional data) because of which we weren't able to understand that when we go deeper with more combinations of different categorical features the polynomials change.
- Also, having a periodical encoding for features might have helped retain that cyclical sorting information within data and might have helped improve performance. Lets make a dummy model without periodical encoding to observe the difference.

### For economy class without cyclical encoding

To avoid too much computational complexity, I will only perform random search cv with 25 iterations and also, the splitter is sent to random.

#### Pre-processing

In [ ]:
eco_wo_cyclic_encode_ct = ColumnTransformer([('Cube_root', PowerTransformer(1/3), cube_root_cols),
                                              ('Passthrough_num_cols', 'passthrough', passthrough_num_cols),
                                              ('Ordinal_encoding', OrdinalEncoder(categories = oridnal_cols_cat_in_order + periodic_cols_cat_in_order, dtype = int), ordinal_cols + periodic_cols),
                                              ('One_hot_encoding', OneHotEncoder(drop = 'first', sparse_output = False, dtype = int), one_hot_encoding_cols)])

X_train_eco_reg_transformed_wo_cyclic_encode = eco_wo_cyclic_encode_ct.fit_transform(X_train_eco_reg)

eco_wo_cyclical_encode_column_names = np.concatenate([[f'cube_root_{col}' for col in cube_root_cols],
                                                       passthrough_num_cols,
                                                       ordinal_cols + periodic_cols,
                                                       eco_wo_cyclic_encode_ct.named_transformers_['One_hot_encoding'].get_feature_names_out()])

X_train_eco_reg_transformed_wo_cyclic_encode = pd.DataFrame(X_train_eco_reg_transformed_wo_cyclic_encode, columns = eco_wo_cyclical_encode_column_names)

#### Perform random search

In [ ]:
model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123)

param_grid = {'max_features': [None, 'sqrt', 'log2'],
              'min_samples_leaf': list(range(10, 101,10)),
              'max_depth' : list(range(1, 21, 2)),
              'criterion' :["squared_error", "friedman_mse", "absolute_error", "poisson"]}

random_cv = RandomizedSearchCV(model, param_distributions = param_grid, n_iter = 25, cv = 3, verbose = True, n_jobs = -1, scoring = 'r2' , return_train_score = False, random_state = 123)

best_random_cv = random_cv.fit(X_train_eco_reg_transformed_wo_cyclic_encode, Y_train_eco_reg)

In [ ]:
best_random_cv.best_params_

#### Check overfitting

In [ ]:
model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123)
model.set_params(**best_random_cv.best_params_)

model_cv_result = cross_validate(model, X_train_eco_reg_transformed_wo_cyclic_encode, Y_train_eco_reg, scoring='r2', cv=3, n_jobs=-1, verbose=0, return_train_score=True, return_estimator = False)

In [ ]:
eco_model_results_df = update_results_df(eco_model_results_df, model_cv_result, 'Decision Tree w/o cyclical encoding')

In [ ]:
eco_model_results_df

**Observations:**  
1. The model performance has slightly decreased. Thus, retaining the cyclical information in the dataset has helped in improving the performance.

### For business class

#### Perform random search

In [ ]:
model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123)

param_grid = {'max_features': [None, 'sqrt', 'log2'],
              'min_samples_leaf': list(range(10, 101,10)),
              'max_depth' : list(range(1, 21, 2)),
              'criterion' :["squared_error", "friedman_mse", "absolute_error", "poisson"]}

random_cv = RandomizedSearchCV(model, param_distributions = param_grid, n_iter = 25, cv = 3, verbose = True, n_jobs = -1, scoring = 'r2' , return_train_score = False, random_state = 123)

best_random_cv = random_cv.fit(X_train_bus_reg_transformed, Y_train_bus_reg)

In [ ]:
best_random_cv.best_params_

#### Check overfitting

In [ ]:
model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123)
model.set_params(**best_random_cv.best_params_)

model_cv_result = cross_validate(model, X_train_bus_reg_transformed, Y_train_bus_reg, scoring='r2', cv=3, n_jobs=-1, verbose=0, return_train_score=True, return_estimator = False)

In [ ]:
bus_model_results_df = pd.DataFrame({'Model':['Decision Tree'], 'Mean train score': [np.mean(model_cv_result['train_score'])], 'Mean test score':[np.mean(model_cv_result['test_score'])]})

In [ ]:
bus_model_results_df

#### Train model on full dataset

In [ ]:
bus_decision_tree_model = tree.DecisionTreeRegressor(splitter = 'random', random_state = 123).set_params(**best_random_cv.best_params_)
bus_decision_tree_model.fit(X_train_bus_reg_transformed, Y_train_bus_reg)

#### Decision tree visualization

In [ ]:
plot_decision_tree(bus_decision_tree_model, X_train_bus_reg_transformed.columns, 'Bus_decision_tree')

**Observations:**  
- Same observations that we made for economy model can be made here as well.

## Random Forest Regressor

I have done manual adjustment of parameters to reach some sub-optimal combination. Ideally, random search and grid search cv should be performed, in case computational cost is not a problem.

### For economy class

#### Cross-validation

In [ ]:
model = RandomForestRegressor(n_estimators = 500,
                            criterion = "squared_error",
                            max_depth = 25,
                            max_features = 'sqrt',
                            max_samples = 0.5,
                            min_samples_leaf = 6,
                            bootstrap = True,
                            oob_score = False,
                            n_jobs = -1,
                            random_state = 123)

model_cv_result = cross_validate(model, X_train_eco_reg_transformed, Y_train_eco_reg, scoring = 'r2', cv = 3, n_jobs = -1, verbose = 0, return_train_score = True, return_estimator = False)

In [ ]:
eco_model_results_df = update_results_df(eco_model_results_df, model_cv_result, 'Random Forest')

In [ ]:
eco_model_results_df

#### Train model on full dataset

In [ ]:
eco_random_forest_model = RandomForestRegressor(n_estimators = 500,
                            criterion = "squared_error",
                            max_depth = 25,
                            max_features = 'sqrt',
                            max_samples = 0.5,
                            min_samples_leaf = 6,
                            bootstrap = True,
                            oob_score = False,
                            n_jobs = -1,
                            random_state = 123)

eco_random_forest_model.fit(X_train_eco_reg_transformed, Y_train_eco_reg)

#### Feature importance

In [ ]:
sns.barplot(x = eco_random_forest_model.feature_importances_, y = eco_random_forest_model.feature_names_in_, orient = 'h')
plt.ylabel('Feature')
plt.xlabel('Feature importance')
plt.show();

**Observations:**  
1. The predictive power of random forest model is better than that of decision tree.  
2. Using an ensemble model, we have removed the inherent high variance problem within decision trees.  
3. Looking at the feature importance, we can observe days left has very high importance followed by cube root duration, stops and then airline being vistara or not(this means vistara is the most distinguished brand in terms of pricing). This we saw during EDA as well where we observed that days left has much more explanatory power compared to cube root duration for economy class. 

### For business class

#### Cross-validation

In [ ]:
model = RandomForestRegressor(n_estimators = 500,
                            criterion = "squared_error",
                            max_depth = 25,
                            max_features = 'sqrt',
                            max_samples = 0.5,
                            min_samples_leaf = 6,
                            bootstrap = True,
                            oob_score = False,
                            n_jobs = -1,
                            random_state = 123)

model_cv_result = cross_validate(model, X_train_bus_reg_transformed, Y_train_bus_reg, scoring = 'r2', cv = 3, n_jobs = -1, verbose = 0, return_train_score = True, return_estimator = False)

In [ ]:
bus_model_results_df = update_results_df(bus_model_results_df, model_cv_result, 'Random Forest')

In [ ]:
bus_model_results_df

#### Train model on full dataset

In [ ]:
bus_random_forest_model = RandomForestRegressor(n_estimators = 500,
                            criterion = "squared_error",
                            max_depth = 25,
                            max_features = 'sqrt',
                            max_samples = 0.5,
                            min_samples_leaf = 6,
                            bootstrap = True,
                            oob_score = False,
                            n_jobs = -1,
                            random_state = 123)

bus_random_forest_model.fit(X_train_bus_reg_transformed, Y_train_bus_reg)

#### Feature importance

In [ ]:
sns.barplot(x = bus_random_forest_model.feature_importances_, y = bus_random_forest_model.feature_names_in_, orient = 'h')
plt.ylabel('Feature')
plt.xlabel('Feature importance')
plt.show();

**Observations:**  
1. The predictive power of random forest model is better than that of decision tree.  
2. Using an ensemble model, we have removed the inherent high variance problem within decision trees.  
3. Looking at the feature importance, we can observe that cube root duration has very high importance followed by stops and then airline being vistara or not. Since for business class we only have two airlines: vistara and air india, this means that airline plays more role in determing price than any other nominal categorical variable. During EDA, we observed that for business class days left doesn't have much importance, only when the days left are very few then it makes some contribution and same thing is reflected here.

## Ada Boost

### For economy class

#### Cross-validation

In [ ]:
model = AdaBoostRegressor(estimator = tree.DecisionTreeRegressor(max_depth = 18,
                                                                 min_samples_leaf = 15,
                                                                 splitter = 'best',
                                                                 criterion = "squared_error",
                                                                 max_features = 'sqrt',
                                                                 random_state = 123),
                          n_estimators = 100,
                          learning_rate = 0.01,
                          random_state = 123)

model_cv_result = cross_validate(model, X_train_eco_reg_transformed, Y_train_eco_reg, scoring = 'r2', cv = 3, n_jobs = -1, verbose = 0, return_train_score = True, return_estimator = False)

In [ ]:
eco_model_results_df = update_results_df(eco_model_results_df, model_cv_result, 'Ada Boost')

In [ ]:
eco_model_results_df

#### Train model on full dataset

In [ ]:
eco_ada_boost_model = AdaBoostRegressor(estimator = tree.DecisionTreeRegressor(max_depth = 18,
                                                                 min_samples_leaf = 15,
                                                                 splitter = 'best',
                                                                 criterion = "squared_error",
                                                                 max_features = 'sqrt',
                                                                 random_state = 123),
                                          n_estimators = 100,
                                          learning_rate = 0.01,
                                          random_state = 123)

eco_ada_boost_model.fit(X_train_eco_reg_transformed, Y_train_eco_reg)

#### Feature importance

In [ ]:
sns.barplot(x = eco_ada_boost_model.feature_importances_, y = eco_ada_boost_model.feature_names_in_, orient = 'h')
plt.ylabel('Feature')
plt.xlabel('Feature importance')
plt.show();

**Observations:**  
1. The predictive power of ada boost model is slightly lower than that of random forest. But we are able to achieve this with so much less number of decision trees in ada boost(100) compared to random forest(500). This is possible due to the corrective nature of ada boosts algorithm where it tries to correct it's mistakes sequentially.  
2. Looking at the feature importance, it looks similar to what we observed for random forest.

### For business class

#### Cross-validation

In [ ]:
model = AdaBoostRegressor(estimator = tree.DecisionTreeRegressor(max_depth = 18,
                                                                 min_samples_leaf = 15,
                                                                 splitter = 'best',
                                                                 criterion = "squared_error",
                                                                 max_features = 'sqrt',
                                                                 random_state = 123),
                          n_estimators = 100,
                          learning_rate = 0.01,
                          random_state = 123)

model_cv_result = cross_validate(model, X_train_bus_reg_transformed, Y_train_bus_reg, scoring = 'r2', cv = 3, n_jobs = -1, verbose = 0, return_train_score = True, return_estimator = False)

In [ ]:
bus_model_results_df = update_results_df(bus_model_results_df, model_cv_result, 'Ada Boost')

In [ ]:
bus_model_results_df

#### Train model on full dataset

In [ ]:
bus_ada_boost_model = AdaBoostRegressor(estimator = tree.DecisionTreeRegressor(max_depth = 18,
                                                                 min_samples_leaf = 15,
                                                                 splitter = 'best',
                                                                 criterion = "squared_error",
                                                                 max_features = 'sqrt',
                                                                 random_state = 123),
                                          n_estimators = 100,
                                          learning_rate = 0.01,
                                          random_state = 123)

bus_ada_boost_model.fit(X_train_bus_reg_transformed, Y_train_bus_reg)

#### Feature importance

In [ ]:
sns.barplot(x = bus_ada_boost_model.feature_importances_, y = bus_ada_boost_model.feature_names_in_, orient = 'h')
plt.ylabel('Feature')
plt.xlabel('Feature importance')
plt.show();

**Observations:**  
1. The predictive power of ada boost model is slightly lower than that of random forest. But we are able to achieve this with so much less number of decision trees in ada boost(100) compared to random forest(500). This is possible due to the corrective nature of ada boosts algorithm where it tries to correct it's mistakes sequentially.  
2. Looking at the feature importance, it looks similar to what we observed for random forest.

## XGBoost

### For economy class

#### Perform Random Search

In [ ]:
model = XGBRegressor(objective='reg:squarederror', silent=True, seed = 123, error_score='raise')

param_grid = {'gamma': np.logspace(-3.2, -2, 10),
              'learning_rate' : np.logspace(-3, 0, 10), 
              'n_estimators' : list(range(200, 501, 50)), 
              'subsample': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
              'lambda': np.logspace(-5, 4, 10),
              'colsample_bytree': [0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
              'max_depth': [21, 23, 25]}

random_cv = RandomizedSearchCV(model, param_distributions = param_grid, n_iter = 50, cv = 3, verbose = True, n_jobs = -1, scoring = 'r2' , return_train_score = False, random_state = 123)

best_random_cv = random_cv.fit(X_train_eco_reg_transformed, Y_train_eco_reg)

In [ ]:
best_random_cv.best_params_

#### Check overfitting

In [ ]:
model = XGBRegressor(objective='reg:squarederror', silent=True, seed = 123, error_score='raise')
model.set_params(**best_random_cv.best_params_)

model_cv_result = cross_validate(model, X_train_eco_reg_transformed, Y_train_eco_reg, scoring='r2', cv=3, n_jobs=-1, verbose=0, return_train_score=True, return_estimator = False)

In [ ]:
eco_model_results_df = update_results_df(eco_model_results_df, model_cv_result, 'XGBoost')

In [ ]:
eco_model_results_df

Little bit of overfitting. This can be made better by doing grid search around the obtained parameters.

#### Train model on full dataset

In [ ]:
eco_xgboost_model = XGBRegressor(objective='reg:squarederror', silent=True, seed = 123, error_score='raise', importance_type = 'total_gain').set_params(**best_random_cv.best_params_)
eco_xgboost_model.fit(X_train_eco_reg_transformed, Y_train_eco_reg)

#### Feature importance

In [ ]:
sns.barplot(x = eco_xgboost_model.feature_importances_, y = eco_xgboost_model.feature_names_in_, orient = 'h')
plt.ylabel('Feature')
plt.xlabel('Feature importance')
plt.show();

**Observations:**  
1. The predictive power of xgboost model is best till now.  
2. Looking at the feature importancte, it looks similar ot what we observed for random forest.

### For business class

#### Perform Random Search

In [ ]:
model = XGBRegressor(objective='reg:squarederror', silent=True, seed = 123, error_score='raise')

param_grid = {'gamma': np.logspace(-4, -2, 10),
              'learning_rate' : np.logspace(-3, 0, 10), 
              'n_estimators' : list(range(200, 501, 50)), 
              'subsample': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
              'lambda': np.logspace(-5, 4, 10),
              'colsample_bytree': [0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
              'max_depth': [21, 23, 25]}

random_cv = RandomizedSearchCV(model, param_distributions = param_grid, n_iter = 50, cv = 3, verbose = True, n_jobs = -1, scoring = 'r2' , return_train_score = False, random_state = 123)

best_random_cv = random_cv.fit(X_train_bus_reg_transformed, Y_train_bus_reg)

In [ ]:
best_random_cv.best_params_

#### Check overfitting

In [ ]:
model = XGBRegressor(objective='reg:squarederror', silent=True, seed = 123, error_score='raise')
model.set_params(**best_random_cv.best_params_)

model_cv_result = cross_validate(model, X_train_bus_reg_transformed, Y_train_bus_reg, scoring='r2', cv=3, n_jobs=-1, verbose=0, return_train_score=True, return_estimator = False)

In [ ]:
bus_model_results_df = update_results_df(bus_model_results_df, model_cv_result, 'XGBoost')

In [ ]:
bus_model_results_df

#### Train model on full dataset

In [ ]:
bus_xgboost_model = XGBRegressor(objective='reg:squarederror', silent=True, seed = 123, error_score='raise', importance_type = 'total_gain').set_params(**best_random_cv.best_params_)
bus_xgboost_model.fit(X_train_bus_reg_transformed, Y_train_bus_reg)

#### Feature importance

In [ ]:
sns.barplot(x = bus_xgboost_model.feature_importances_, y = bus_xgboost_model.feature_names_in_, orient = 'h')
plt.ylabel('Feature')
plt.xlabel('Feature importance')
plt.show();

**Observations:**  
1. The predictive power of xgboost model is best that we have seen till now.  
2. Looking at the feature importance, it looks similar ot what we observed for random forest.

## Voting Regressor

### For Economy class

In [ ]:
decision_tree_estimator = tree.DecisionTreeRegressor(min_samples_leaf = 20, 
                                                     max_features = None, 
                                                     max_depth = 17, 
                                                     criterion = 'poisson', 
                                                     splitter = 'random', 
                                                     random_state = 123)

random_forest_estimator =  RandomForestRegressor(n_estimators = 500,
                                                criterion = "squared_error",
                                                max_depth = 25,
                                                max_features = 'sqrt',
                                                max_samples = 0.5,
                                                min_samples_leaf = 6,
                                                bootstrap = True,
                                                oob_score = False,
                                                n_jobs = -1,
                                                random_state = 123)

adaboost_estimator = AdaBoostRegressor(estimator = tree.DecisionTreeRegressor(max_depth = 18,
                                                                 min_samples_leaf = 15,
                                                                 splitter = 'best',
                                                                 criterion = "squared_error",
                                                                 max_features = 'sqrt',
                                                                 random_state = 123),
                                      n_estimators = 100,
                                      learning_rate = 0.01,
                                      random_state = 123)

xgboost_estimator = XGBRegressor(subsample = 0.9, n_estimators = 400, max_depth = 21, 
                                 learning_rate = 0.021544346900318832, reg_lambda = 0.01, 
                                 gamma = 0.000630957344480193, colsample_bytree = 0.7, 
                                 objective='reg:squarederror', silent=True,
                                 seed = 123, error_score='raise')

model = VotingRegressor([('DECISION_TREE', decision_tree_estimator), 
                         ('RANDOM_FOREST', random_forest_estimator),
                         ('ADABOOST', adaboost_estimator),
                         ('XGBOOST', xgboost_estimator)], n_jobs = -1)

model_cv_result = cross_validate(model, X_train_eco_reg_transformed, Y_train_eco_reg, scoring = 'r2', cv = 3, n_jobs = -1, verbose = 0, return_train_score = True, return_estimator = False)

In [ ]:
eco_model_results_df = update_results_df(eco_model_results_df, model_cv_result, 'Voting Regressor')

In [ ]:
eco_model_results_df

### For Business class

In [ ]:
decision_tree_estimator = tree.DecisionTreeRegressor(min_samples_leaf = 20, 
                                                     max_features = None, 
                                                     max_depth = 17, 
                                                     criterion = 'poisson', 
                                                     splitter = 'random', 
                                                     random_state = 123)

random_forest_estimator =  RandomForestRegressor(n_estimators = 500,
                                                criterion = "squared_error",
                                                max_depth = 25,
                                                max_features = 'sqrt',
                                                max_samples = 0.5,
                                                min_samples_leaf = 6,
                                                bootstrap = True,
                                                oob_score = False,
                                                n_jobs = -1,
                                                random_state = 123)

adaboost_estimator = AdaBoostRegressor(estimator = tree.DecisionTreeRegressor(max_depth = 18,
                                                                 min_samples_leaf = 15,
                                                                 splitter = 'best',
                                                                 criterion = "squared_error",
                                                                 max_features = 'sqrt',
                                                                 random_state = 123),
                                      n_estimators = 100,
                                      learning_rate = 0.01,
                                      random_state = 123)

xgboost_estimator = XGBRegressor(subsample = 0.9, n_estimators = 400, max_depth = 21, 
                                 learning_rate = 0.021544346900318832, reg_lambda = 0.01, 
                                 gamma = 0.0001, colsample_bytree = 0.7, 
                                 objective='reg:squarederror', silent=True,
                                 seed = 123, error_score='raise')

model = VotingRegressor([('DECISION_TREE', decision_tree_estimator), 
                         ('RANDOM_FOREST', random_forest_estimator),
                         ('ADABOOST', adaboost_estimator),
                         ('XGBOOST', xgboost_estimator)], n_jobs = -1)

model_cv_result = cross_validate(model, X_train_bus_reg_transformed, Y_train_bus_reg, scoring = 'r2', cv = 3, n_jobs = -1, verbose = 0, return_train_score = True, return_estimator = False)

In [ ]:
bus_model_results_df = update_results_df(bus_model_results_df, model_cv_result, 'Voting Regressor')

In [ ]:
bus_model_results_df

## Deep Learning

### For Economy class

#### Model Building

Through some manual iterations(due to computational constraints) and optimizing learning rate, I was able to find these parameters as most suitable with no overfitting. 

In [ ]:
# setting seed for reproducibility
tf.random.set_seed(123)
np.random.seed(123)

# hyper parameters
number_of_neurons = 40
number_of_layers = 5
batch_size = 32
no_of_epochs = 200
learning_rate = 0.001291549665014884

#norm layer
eco_norm_layer = tf.keras.layers.Normalization(axis = -1)

#defining the models: input, norm and first hidden layer
eco_model = tf.keras.Sequential([tf.keras.layers.Input(shape = X_train_eco_reg_transformed.shape[1:]), 
                                 eco_norm_layer])

#adding hidden layers
for i in range(number_of_layers):
    eco_model.add(tf.keras.layers.Dense(number_of_neurons, activation = 'relu', kernel_initializer = 'he_uniform'))

#adding ouput layer for regression with 1 node and activation function as linear
eco_model.add(tf.keras.layers.Dense(1, activation = 'linear', kernel_initializer = 'he_uniform'))

#compiling the model
eco_model.compile(optimizer = keras.optimizers.Adam(learning_rate = learning_rate),
                  loss = "mse", 
                  metrics = [keras.metrics.R2Score()])

#callbacks
early_stopping_cb = tf.keras.callbacks.EarlyStopping(monitor = 'val_r2_score', patience = 25, restore_best_weights = True)
print_diff_cb = PrintValTrainDiffCallback()
tensorboard_cb = tf.keras.callbacks.TensorBoard(get_run_logdir(number_of_neurons, number_of_layers, learning_rate, batch_size)) 

#adapting norm layer
eco_norm_layer.adapt(X_train_eco_reg_transformed.values)

#fitting the model
eco_history = eco_model.fit(X_train_eco_reg_transformed, Y_train_eco_reg, epochs = no_of_epochs, batch_size = batch_size, validation_split = 0.05, callbacks = [print_diff_cb, early_stopping_cb, tensorboard_cb])

In [ ]:
plot_dl_model_history(eco_history)

In [ ]:
best_epoch = np.argmax(eco_history.history['val_r2_score'])
best_val_score = eco_history.history['val_r2_score'][best_epoch]
best_val_train_score = eco_history.history['r2_score'][best_epoch]

In [ ]:
eco_model_results_df = eco_model_results_df._append({'Model': 'Deep Learning', 'Mean train score': best_val_train_score, 'Mean test score':best_val_score}, ignore_index=True)

In [ ]:
eco_model_results_df

**Observations:**  
1. Till now best performance has been obtained for XGBoost. Deep learning model can be optimized more by trying more parameters using random search and grid search.

### For Business class

#### Model Building

Using same parameters as economy model should give us decent results. This has been seen with other models also.

In [ ]:
# setting seed for reproducibility
tf.random.set_seed(123)
np.random.seed(123)

# hyper parameters
number_of_neurons = 40
number_of_layers = 5
batch_size = 32
no_of_epochs = 200
learning_rate = 0.001291549665014884

#norm layer
bus_norm_layer = tf.keras.layers.Normalization(axis = -1)

#defining the models: input, norm and first hidden layer
bus_model = tf.keras.Sequential([tf.keras.layers.Input(shape = X_train_bus_reg_transformed.shape[1:]), bus_norm_layer])

#adding more hidden layers
for i in range(number_of_layers):
    bus_model.add(tf.keras.layers.Dense(number_of_neurons, activation = 'relu'))

#adding ouput layer for regression with 1 node and activation as linear
bus_model.add(tf.keras.layers.Dense(1, activation = 'linear'))

#compiling the model
bus_model.compile(optimizer = keras.optimizers.Adam(learning_rate = learning_rate),
                  loss = "mse", 
                  metrics = [keras.metrics.R2Score()])

#callbacks
early_stopping_cb = tf.keras.callbacks.EarlyStopping(monitor = 'val_r2_score', patience = 25, restore_best_weights = True)
print_diff_cb = PrintValTrainDiffCallback()
tensorboard_cb = tf.keras.callbacks.TensorBoard(get_run_logdir(number_of_neurons, number_of_layers, learning_rate, batch_size)) 

#adapting norm layer
bus_norm_layer.adapt(X_train_bus_reg_transformed.values)

#fitting the model
bus_history = bus_model.fit(X_train_bus_reg_transformed, Y_train_bus_reg, epochs = no_of_epochs, batch_size = batch_size, validation_split = 0.1, callbacks = [print_diff_cb, early_stopping_cb, tensorboard_cb])

In [ ]:
plot_dl_model_history(bus_history)

In [ ]:
best_epoch = np.argmax(bus_history.history['val_r2_score'])
best_val_score = bus_history.history['val_r2_score'][best_epoch]
best_val_train_score = bus_history.history['r2_score'][best_epoch]

In [ ]:
bus_model_results_df = bus_model_results_df._append({'Model': 'Deep Learning', 'Mean train score': best_val_train_score, 'Mean test score':best_val_score}, ignore_index=True)

In [ ]:
bus_model_results_df

**Observations:**  
1. Till now best performance has been obtained for XGBoost. Deep learning model can be optimized more by trying more parameters using random search and grid search.

## Combined Model

In [ ]:
class combined_model():
    def __init__(self, eco_pipeline, eco_column_names, eco_model, bus_pipeline, bus_column_names, bus_model):
        self.eco_pipeline = eco_pipeline
        self.eco_column_names = eco_column_names
        self.eco_model = eco_model
        self.bus_pipeline = bus_pipeline
        self.bus_column_names = bus_column_names
        self.bus_model = bus_model
        
    def predict(self, X):
        '''Perform prediction using given raw dataset as a dataframe.'''
        X_eco = X[X['class'] == 'Economy'].drop(columns = ['class'])
        X_eco_transformed = pd.DataFrame(self.eco_pipeline.transform(X_eco), index = X_eco.index, columns = self.eco_column_names)
        X_bus = X[X['class'] == 'Business'].drop(columns = ['class'])
        X_bus_transformed = pd.DataFrame(self.bus_pipeline.transform(X_bus), index = X_bus.index, columns = self.bus_column_names)

        Y_eco_transformed_pred = pd.Series(self.eco_model.predict(X_eco_transformed), index = X_eco_transformed.index)
        Y_bus_transformed_pred = pd.Series(self.bus_model.predict(X_bus_transformed), index = X_bus_transformed.index)

        Y_eco_pred = 10**(np.power(Y_eco_transformed_pred, -1))
        Y_bus_pred = (np.power(Y_bus_transformed_pred, -1/0.3))**2

        Y_pred = pd.concat([Y_eco_pred, Y_bus_pred])[X.index]
        return Y_pred

In [ ]:
flight_price_prediction_combined_model = combined_model(eco_pipeline, eco_column_names, eco_xgboost_model, bus_pipeline, bus_column_names, bus_xgboost_model)

## Model Evaluation for price on test set

In [ ]:
Y_train_pred = flight_price_prediction_combined_model.predict(df_train.iloc[:, :-1])

In [ ]:
Y_test_pred = flight_price_prediction_combined_model.predict(df_test.iloc[:, :-1])

In [ ]:
print('For train dataset:')
print('R2 score: ', r2_score(df_train['price'], Y_train_pred))
print('RMSE score: ', np.sqrt(mean_squared_error(df_train['price'], Y_train_pred)))

In [ ]:
print('For test dataset:')
print('R2 score: ', r2_score(df_test['price'], Y_test_pred))
print('RMSE score: ', np.sqrt(mean_squared_error(df_test['price'], Y_test_pred)))

## Fututre Scope:

1. With enough computaional resources, we can do more detailed search of hyper parameters using random search, followed by grid search.  
2. If possible more features like holidays, number of seats left etc. can be gathered which will help explain the pricing more.